# Two staff labelling comments: kappa

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Kappa Coefficient** · a worked example from the StatNex teaching guide

**The story.** Two social-media staff members independently labelled the same 262 customer comments as very positive, positive, negative or very negative. Before automating this with AI, the manager wants to know whether humans even agree.

**The question.** How consistently do our two staff label customer sentiment?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/kappa_coefficient.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Title,Staff A's label,Very positive,Positive,Negative,Very negative,Total
Staff A,Very positive,38.0,9.0,2.0,1.0,50.0
,Positive,12.0,61.0,8.0,3.0,84.0
,Negative,2.0,10.0,52.0,9.0,73.0
,Very negative,1.0,2.0,11.0,41.0,55.0
,Total,53.0,82.0,73.0,54.0,262.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats.contingency import expected_freq
from statsmodels.stats.inter_rater import cohens_kappa

pd.set_option("display.width", 120)

## The data

In [ ]:
# An agreement table: rater A's categories in the rows, rater B's in the columns, in
# the same order, so the diagonal holds the cases on which they agree.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Very positive", "Positive", "Negative", "Very negative"]
table = data.set_index("Staff A's label")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()
counts = table.to_numpy(float)

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## Expected frequency

In [ ]:
# What two raters answering independently, each with their own marginal shares, would
# produce. The expected diagonal is the agreement chance alone gives.
expected = pd.DataFrame(expected_freq(counts), index=table.index, columns=table.columns)
print()
print(expected)

## Weights

In [ ]:
# A near miss earns partial credit: 1 on the diagonal, falling to 0 at the far corner,
# linearly (1 - d) or quadratically (1 - d^2) in the distance d = |i - j| / (k - 1).
size = len(counts)
distance = np.abs(np.subtract.outer(np.arange(size), np.arange(size))) / (size - 1)
linear_weights = pd.DataFrame(1 - distance, index=table.index, columns=table.columns)
quadratic_weights = pd.DataFrame(1 - distance**2, index=table.index, columns=table.columns)
print()
print(linear_weights.round(4))
print()
print(quadratic_weights.round(4))

## Kappa coefficient

In [ ]:
# statsmodels' cohens_kappa gives kappa, its standard error, and the z test of kappa = 0
# (which uses the standard error under independence, std_kappa0). Its own interval is
# fixed at 95%, so the interval is built here from the standard error at the chosen level.
# wt="linear" and wt="quadratic" are the two weightings above.
# How to read it: kappa = (observed agreement - chance agreement) / (1 - chance
# agreement), so 0 is no better than chance and 1 is perfect. The weighted versions give
# partial credit for near misses. The z test only asks whether kappa is above 0, which is
# rarely in doubt; report kappa with its interval.
level = 95 / 100
z_critical = stats.norm.ppf(0.5 + level / 2)
rows = []
for name, wt in (("None", None), ("linear weights", "linear"), ("quadratic weights", "quadratic")):
    result = cohens_kappa(counts, wt=wt)
    rows.append([name, result.kappa, result.std_kappa,
                 result.kappa - z_critical * result.std_kappa,
                 result.kappa + z_critical * result.std_kappa,
                 result.z_value, result.pvalue_two_sided])
kappa = pd.DataFrame(rows, columns=["weighting", "kappa", "SE", "lower", "upper", "z", "P"])
print()
print(f"{level:.0%} confidence interval")
print(kappa.to_string(index=False))

## What the result says

The two staff gave the same label to 38, 61, 52 and 41 comments in the four categories. But some agreement happens by pure chance. The kappa coefficient removes that part: 0 means only chance agreement, 1 means perfect agreement.

Kappa is 0.64, with a standard error of 0.037 and a confidence interval, the range of plausible values, from 0.57 to 0.71. Weighted kappa gives part credit for near misses, such as Positive against Very positive. It is higher: 0.72 with linear weights and 0.80 with quadratic weights. This shows that most disagreements are between neighbouring labels.

So the staff agree fairly well, and mostly disagree about how strong a feeling is, not about whether it is positive or negative.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.